<a href="https://colab.research.google.com/github/Trendtek254/plp-da-a3-duka-sales-and-agent-cleanup/blob/main/duka_sales_exploration.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**Duka Electronics**

***Executive Summary***


1. The dataset contains 1,000 sales transactions across 5 branches.

2. The business sells 5 different product categories.

3. Sales records cover a period of 180 days, from January to June 2026.

4. Customers purchase through 3 sales channels: in-store, online and
phone orders.

5. The dataset contains 7 columns describing transaction dates, branches, products, quantities and prices.


***Task 1: Loading the data***

In [29]:
import pandas as pd
import numpy as np
url = "https://raw.githubusercontent.com/Trendtek254/plp-da-a3-duka-sales-and-agent-cleanup/refs/heads/main/duka_sales.csv"
sales = pd.read_csv(url,parse_dates=["order_date"])
#1.preview the data
sales.head()

#2.Dataset shape
print("Shape:",sales.shape)

#3.Column information
sales.info()

#4.Missing values
print(sales.isna().sum())

#5.Date range
print("Earliest date:",sales["order_date"].min())
print("Latest date:",sales["order_date"].max())

#6.Key categories
print("--------Branches:------")
print(sales["branch"].value_counts())
print("-------Products:-------")
print(sales["product"].value_counts())
print("-------Sales channels:-------")
print(sales["channel"].value_counts())

Shape: (1000, 7)
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 7 columns):
 #   Column      Non-Null Count  Dtype         
---  ------      --------------  -----         
 0   order_id    1000 non-null   int64         
 1   order_date  1000 non-null   datetime64[ns]
 2   branch      1000 non-null   object        
 3   product     1000 non-null   object        
 4   quantity    1000 non-null   int64         
 5   channel     1000 non-null   object        
 6   unit_price  1000 non-null   int64         
dtypes: datetime64[ns](1), int64(3), object(3)
memory usage: 54.8+ KB
order_id      0
order_date    0
branch        0
product       0
quantity      0
channel       0
unit_price    0
dtype: int64
Earliest date: 2026-01-01 00:00:00
Latest date: 2026-06-29 00:00:00
--------Branches:------
branch
Nairobi    376
Mombasa    186
Nakuru     162
Kisumu     159
Eldoret    117
Name: count, dtype: int64
-------Products:-------
product
Laptop        220
T

**First Five Minutes: Dataset Inspection**

***1.Dataset Shape***

-Rows: 1000

-Columns: 7

***2.Data Types***

-Date Column: datetime64[ns]

-Numeric columns: int64   

-Categorical columns: object  

***3.Missing values***

-Columns with missing values:0

-Total missing values:0

***4.Date Range***

-Earliest Date: 2026-01-01 00:00:00

-Latest Date: 2026-06-29 00:00:00 00

***5.Key categories***

-Branches:

-Nairobi :   376

-Mombasa   : 186

-Nakuru   :  162

-Kisumu   :  159

-Eldoret  :  117

-Products:

-Laptop     :   220

-Tablet    :    215

-Phone     :    194

-Headphones :   187

-Charger  :     184

-Sales channels:

-In-store  :     579

-Online    :     320

-Phone order  :  101

***6.Initial Observations***

-The dataset has 1000 rows and 7 columns.

-There are no missing values in the dataset.

-The purchase dates are between 2026-01-01 and 2026-06-29 .

-The sales channels are in-store,online and phone order.

***Task 2: Derrriving columns***

In [30]:
sales["revenue"] = sales["quantity"] * sales["unit_price"]
sales["month"] = sales["order_date"].dt.month_name()
sales["weekday"] = sales["order_date"].dt.day_name()
sales["is_weekend"] = sales["order_date"].dt.dayofweek >= 5
sales["price_band"] = np.where(
    sales["unit_price"] >= 28000,
    "Premium",
    "Value"
)
sales.tail()

,order_id,order_date,branch,product,quantity,channel,unit_price,revenue,month,weekday,is_weekend,price_band
995,1996,2026-03-04,Mombasa,Laptop,5,In-store,65000,325000,March,Wednesday,False,Premium
996,1997,2026-03-06,Nairobi,Tablet,2,Phone order,28000,56000,March,Friday,False,Premium
997,1998,2026-01-20,Nairobi,Laptop,3,Online,65000,195000,January,Tuesday,False,Premium
998,1999,2026-03-02,Nairobi,Headphones,2,In-store,2500,5000,March,Monday,False,Value
999,2000,2026-01-30,Nakuru,Charger,4,In-store,1200,4800,January,Friday,False,Value


***Task 3: Ranking the branches***

In [31]:
branch_performance = sales.groupby("branch").agg(
    total_revenue =("revenue","sum"),
    number_of_orders = ("order_id","count")
)
branch_performance["orders_rank"] = (
    branch_performance["number_of_orders"].rank(
        ascending = False,method ="dense"
    )
)
branch_performance["revenue_rank"] = (
    branch_performance["total_revenue"].rank(
        ascending = False,method ="dense"
    )
)
branch_performance = branch_performance.sort_values("revenue_rank")

branch_performance

,total_revenue,number_of_orders,orders_rank,revenue_rank
branch,,,,
Nairobi,26106100,376,1.0,1.0
Mombasa,14003700,186,2.0,2.0
Nakuru,11970400,162,3.0,3.0
Kisumu,10868100,159,4.0,4.0
Eldoret,8844200,117,5.0,5.0


***The rankings are still the same because the branches that have the highest number of orders are the ones that tend to generate the highest total revenues hence making the rankings the same***

***Task4***

In [32]:
product_revenue = sales.groupby("product").agg(
    total_revenue = ("revenue","sum")
)
top_3_revenue = product_revenue.nlargest(3,"total_revenue")

product_quantity = sales.groupby("product").agg(
    total_units_sold = ("quantity","sum")
)
top_3_units = product_quantity.nlargest(3,"total_units_sold")



***-In the rank list by revenue ,laptops generated the highest revenue followed by tablets and phones the lowest.

-In the rank list by number of units sold,tablets had the highest number of units sold ,followed by laptops while phones had the lowest number of units sold.

-The difference in the two rankings is due to the unit prices by product i.e laptops can generate a high revenue even with small number of units sold compared to tablets due to their high prices per unit.***

***Task 5:***

In [33]:
weekend_comparison = sales.groupby("is_weekend").agg(
    number_of_orders =("order_id","count"),
    total_revenue = ("revenue","sum"),
    average_order_value = ("revenue","mean")
)
weekend_comparison = weekend_comparison.rename(
    index ={
        False: "Weekday",
        True: "Weekend"
    }
)
weekend_comparison

,number_of_orders,total_revenue,average_order_value
is_weekend,,,
Weekday,723,51856200,71723.651452
Weekend,277,19936300,71972.202166


***Weekend vs Weekday Sales Comparison***

-Weekdays recorded 723 orders,while weekends recorded 277 orders.

-Weekday revenue was KES 51,856,200 ,compared with KES 19,936,300 during weekends.

-The average order was KES 71,723.65 on weekdays and KES 71,972.20 on weekends.

 ***Interpretation:***

 The weekdays performed better compared to the weekend.This shows that most customers make purchases during weekdays.





***Task 6:***

In [34]:
sales["month"] = sales["order_date"].dt.to_period("M")
monthly_revenue = sales.groupby("month")["revenue"].sum()
best_month = monthly_revenue.idxmax()
worst_month = monthly_revenue.idxmin()

best_revenue = monthly_revenue.max()
worst_revenue = monthly_revenue.min()

percentage_diff = (best_revenue - worst_revenue) / worst_revenue * 100

print(f"Best Month: {best_month} with Revenue KES {best_revenue:,.2f}")
print(f"Worst Month: {worst_month} with Revenue KES {worst_revenue:,.2f}")
print(f"Percentage difference: {percentage_diff:.2f}%")

Best Month: 2026-04 with Revenue KES 13,518,300.00
Worst Month: 2026-06 with Revenue KES 10,903,400.00
Percentage difference: 23.98%


***Task 7:***
****Filtering Premium Online orders from Nairobi and Mombasa****

In [43]:
premium_online = sales[
    (sales["price_band"] == "Premium") &
    (sales["channel"] == "Online") &
    (sales["branch"].isin(["Nairobi", "Mombasa"]))
].copy()

number_of_orders = len(premium_online)

total_revenue = premium_online["revenue"].sum()

print("Number of orders:", number_of_orders)
print("Total revenue: KES", total_revenue)
premium_online.to_csv(
    "premium_online_coast_capital.csv",
    index=False
)



Number of orders: 83
Total revenue: KES 10218000


In [44]:
pandas_total = sales["revenue"].sum()

print("Pandas total revenue:", pandas_total)

Pandas total revenue: 71792500


In [45]:
sheets_total = 71792500 # Replace with your actual Sheets total

print("Pandas:", pandas_total)
print("Google Sheets:", sheets_total)

if pandas_total == sheets_total:
    print("SUCCESS: Totals match!")
else:
    print("CHECK: Totals do not match.")

Pandas: 71792500
Google Sheets: 71792500
SUCCESS: Totals match!
